# Recipe Chat Bot
### Start installing needed packages
### mounting your drive if needed


In [1]:
!pip install -q langchain langchain-community langchain-chroma
!pip install -q langchain_openai langchain-classic
!pip install -q langchain-huggingface huggingface_hub
!pip install -q langchain-text-splitters pypdf pymupdf chromadb
# Force-upgrade bitsandbytes and accelerate to meet minimum version requirements
!pip install -U -q "bitsandbytes>=0.46.1" accelerate

from google.colab import drive
drive.mount('/content/drive')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=

# Load PDFs from Google Drive using PyMuPDF

In [2]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
#from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

PDF_FOLDER = "/content/drive/MyDrive/Recipe_RAG/recipe_pdfs"
DB_FOLDER = "/content/drive/MyDrive/Recipe_RAG/chroma_recipes_db"

# 1. Load PDFs from Google Drive using PyMuPDF
# We pass `silent_errors=True` to DirectoryLoader to avoid crash on bad files
loader = PyPDFDirectoryLoader(PDF_FOLDER, glob="**/*.pdf",
                              silent_errors=True)
#loader = DirectoryLoader(
 #   PDF_FOLDER,
 #   glob="**/*.pdf",
  #  loader_cls=PyMuPDFLoader,
  #  show_progress=True,
 #   silent_errors=True
#)
try:
  docs = loader.load()
  print(f"Successfully loaded {len(docs)} pages/documents!")
except Exception as e:
    print(f"An error occurred during loading: {e}")

/tmp/ipykernel_1813/1162027745.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Successfully loaded 1976 pages/documents!


# Split into recipe-sized chunks


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Split into recipe-sized chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1800,
    chunk_overlap=250,
    separators=["\n\n", "\n", ".", " "]
)
chunks = text_splitter.split_documents(docs)
print(f"{len(chunks)} chunks created")

2953 chunks created


#### Print sample chunks to check how they split

In [4]:
# Inspect Chunk 10 and Chunk 11 (skipping early table-of-contents pages)

for i in range(10, 12):
    print(f"================ CHUNK {i} ================")
    print(f"Length: {len(chunks[i].page_content)} characters")
    print(f"Metadata: {chunks[i].metadata}")
    print("-" * 40)
    print(chunks[i].page_content)
    print("\n")

================ CHUNK 10 ================
Length: 72 characters
Metadata: {'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2012-09-13T17:06:20-04:00', 'moddate': '2012-09-13T17:10:17-04:00', 'trapped': '/False', 'source': '/content/drive/MyDrive/Recipe_RAG/recipe_pdfs/MyPlate_RecipeBook2012_FINAL_SinglePages-5B8E06286B084B78B2EA5174E1711C5F.pdf', 'total_pages': 64, 'page': 9, 'page_label': '10'}
----------------------------------------
Preparation Time: 10 minutes
clementine, raisin & Goat cheese Sandwich
7


================ CHUNK 11 ================
Length: 1414 characters
Metadata: {'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2012-09-13T17:06:20-04:00', 'moddate': '2012-09-13T17:10:17-04:00', 'trapped': '/False', 'source': '/content/drive/MyDrive/Recipe_RAG/recipe_pdfs/MyPlate_RecipeBook2012_FINAL_SinglePages-5B8E06286B084B78B2EA5174E1711C5F.pdf', 'total_pages': 64, 'page

# Create embeddings and store in ChromaDB
### persist it on the drive

In [5]:
import os
import torch
from google.colab import userdata

# Store Hugging face read token in the env safely in Colab's left sidebar (Key icon -> Secrets)
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print (f"device we will use = {device}")


device we will use = cuda


   ##### Details at https://huggingface.co/Qwen/Qwen3-Embedding-0.6B
   * Accuracy improves by 1-5% on Qwen3-Embedding models as it is instruction-aware by adding query_encode_kwargs.
   * It helps to treat Queries and Documents Differently as querys are short than chunks.
   * Adding "Instruct: ...\nQuery: " makes LangChain attach it when embedding the user's question and not the chunks.
   * This helps the model to map the question in same vector space neighborhood as full recipe passages.

    

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Pass the detected device into HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"device": device,
        "model_kwargs": {"torch_dtype": torch.float16} # Cuts memory usage in half
    },
    encode_kwargs={
        "batch_size": 32, # Prevents out-of-memory spikes on large datasets
        "normalize_embeddings": True
    },
    # Read details in the text cell above
    query_encode_kwargs={
        "prompt": "Instruct: Given a cooking question or dish request, retrieve relevant recipe passages with ingredients and instructions.\nQuery: ",
        "normalize_embeddings": True
    }
)

# uncomment following code if need to regenerate the vectors
# remove the old saved from drive else double chunks will be created

# 3. Create and save Chroma DB permanently to Google Drive
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_FOLDER
)
print(f"Saved {len(chunks)} recipe chunks to Google Drive!")

# uncomment following code to load vectors from saved drive
"""
# 3. Load saved Chroma DB from Google Drive
DB_FOLDER = "/content/drive/MyDrive/Recipe_RAG/chroma_recipes_db"

vectorstore = Chroma(
    persist_directory=DB_FOLDER,
    embedding_function=embeddings
)

# 3. Verify that all chunks loaded properly
print(f"Loaded {vectorstore._collection.count()} chunks from Google Drive!")
"""

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Saved 2953 recipe chunks to Google Drive!


'\n# 3. Load saved Chroma DB from Google Drive\nDB_FOLDER = "/content/drive/MyDrive/Recipe_RAG/chroma_recipes_db"\n\nvectorstore = Chroma(\n    persist_directory=DB_FOLDER,\n    embedding_function=embeddings\n)\n\n# 3. Verify that all chunks loaded properly\nprint(f"Loaded {vectorstore._collection.count()} chunks from Google Drive!")\n'

# Build RAG chain for QA with the bot

In [7]:
import torch
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Configure 4-bit quantization to fit the model in limited VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

# Ensure Qwen stops immediately at <|im_end|> instead of generating extra tokens
stop_ids = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|im_end|>")
]

# 1. Create native Transformers Pipeline
hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=600,
    do_sample=False,           # Greedy decoding
    repetition_penalty=1.1,
    eos_token_id=stop_ids,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False
)

# 2. Wrap in LangChain's HuggingFacePipeline first!
local_llm = HuggingFacePipeline(pipeline=hf_pipe)

# 3. Now wrap in ChatHuggingFace so it formats as chat messages properly
llm = ChatHuggingFace(llm=local_llm)

# 4. Define the custom cooking PromptTemplate (must use {context} and {question})
recipe_template = """You are an expert chef and helpful recipe assistant.
Answer the user's question using ONLY the recipe context provided below.
Do not invent ingredients or steps that are not in the context.
If the requested recipe is not in the context, reply: "I couldn't find that recipe in your uploaded cookbooks."

When providing a recipe, format your response clearly as:
- **Recipe Name**
- **Prep / Cook Time** (if mentioned)
- **Ingredients** (bulleted list)
- **Instructions** (numbered steps)

Recipe Context:
{context}

User Question: {question}"""

QA_PROMPT = PromptTemplate(
    template=recipe_template,
    input_variables=["context", "question"]
)

# 5. Connect the custom prompt into RetrievalQA via chain_type_kwargs
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_PROMPT}
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

### Query the recipe bot with a sample question

In [8]:
# 6. Test your recipe bot

result = qa_chain.invoke({"query": "Give me a bread recipe from the loaded PDFs."})
print("🤖 Bot Response:\n")
print(result["result"])
#print("\nSources used:")
#for doc in result["source_documents"]:
#    print(f"- File: {doc.metadata.get('source')} | Page: {doc.metadata.get('page')}")

🤖 Bot Response:

**Title: Golden Raisin Bread**

- **Servings:** 1
- **Ingredients:**
  - 1 pkg Yeast
  - 2 tbsp Soft butter
  - 2 cups Bread flour
  - 3/4 cup Golden raisins
  - 1/4 cup Whole wheat flour
  - 1/4 tsp Ground ginger
  - 1/4 tsp Cinnamon
  - 1/2 cup Brown sugar
  - 1 1/2 cups Warm water
  - 1 tsp Salt

- **Instructions:**
  1. In a pan, combine all the ingredients in the order listed.
  2. Select "white bread" and press "start."
  3. Follow the instructions given by the meal-master software.

This recipe uses a breadmaker to prepare the golden raisin bread.


In [9]:
# 5. Query!
result = qa_chain.invoke(
    "How do you make a Clementine, Raisin & Goat Cheese Sandwich?"
)
print("🤖 Bot Response:\n")
print(result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print(f"  Source: {doc.metadata.get('source', 'Unknown')}, p.{doc.metadata.get('page', 'Unknown')}")

🤖 Bot Response:

**Clementine, Raisin & Goat Cheese Sandwich**

- **Prep / Cook Time**: 10 minutes

- **Ingredients**:
  - 4 Dandy clementine oranges
  - 1 cup Sun-Maid raisins
  - 4 oz (1 cup) crumbled goat cheese (or feta cheese)
  - 3 tbsp chopped walnuts
  - 2 tbsp fat-free yogurt, plain
  - 2 tbsp fat-reduced mayonnaise
  - 8 lettuce leaves (Bibb, Romaine, red, or green leaf)
  - 8 slices whole grain bread

- **Instructions**:
  1. Peel clementine oranges and split each in half. Chop the orange halves into small pieces.
  2. In a medium bowl, mix raisins, cheese, walnuts, yogurt, and mayonnaise. Add chopped clementine oranges and mix gently.
  3. Lay out bread slices. Spread ¼ lettuce mix on 4 of the bread slices. Add two lettuce leaves to each sandwich; top with 2nd bread slice.
  4. Serves 4.

Nutrition Information per Serving:
- Calories: 440
- Total Fat: 15g
- Saturated Fat: 5g
- % of Calories from Fat: 31%
- % Calories from Sat Fat: 10%
- Protein: 16g
- Carbohydrates: 67g
- C

# Set up interactive demo using Gradio in colab

In [10]:
!pip install -q gradio

import os
import gradio as gr

def chat_with_chef(message, history):
    # Query your existing RetrievalQA chain
    response = qa_chain.invoke({"query": message})
    answer = response["result"].strip()

    # Format the PDF sources cleanly below the recipe
    sources = []
    for doc in response.get("source_documents", []):
        file_name = os.path.basename(doc.metadata.get("source", "Unknown PDF"))
        page_num = doc.metadata.get("page", 0) + 1
        source_tag = f"{file_name} (Page {page_num})"
        if source_tag not in sources:
            sources.append(source_tag)

    if sources:
        answer += "\n\n---\n**Sources:** " + ", ".join(sources)

    return answer

# Launch the Gradio ChatInterface inside Colab
demo = gr.ChatInterface(
    fn=chat_with_chef,
    title="🍳 Cookbook RAG Recipe Assistant",
    description="Ask for recipes, ingredients, or cooking instructions from uploaded PDF cookbooks.",
    examples=[
        "Give me a adrak chai recipe from the loaded PDFs.",
        "What vegetarian dinner recipes do you have?",
        "How do I make pasta sauce?"
    ]
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cda6bdf9b79db3ec30.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
